# Практикум 12. K-Means

Сопровождает интерактивную лабораторию №12 («Алгоритм K-Means»). Тема курса: автоматическая группировка текстов и влияние параметров

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
vec = TfidfVectorizer(); X = vec.fit_transform(df.text); y = df.label.values
terms = np.array(vec.get_feature_names_out())

## 1. Кластеры и их ключевые слова

In [ ]:
km = KMeans(4, init="k-means++", n_init=10, random_state=0).fit(X)
print("ARI:", round(adjusted_rand_score(y, km.labels_), 3), "| inertia:", round(km.inertia_, 2))
for c in range(4):
    print(f"Кластер {c}:", ", ".join(terms[km.cluster_centers_[c].argsort()[::-1][:6]]), "|", pd.Series(y[km.labels_ == c]).value_counts().to_dict())

## 2. Инициализация: локальные минимумы
Один запуск (`n_init=1`) с разными seed: случайная и k-means++ инициализация.

In [ ]:
res = {}
for init in ["random", "k-means++"]:
    res[init] = [KMeans(4, init=init, n_init=1, random_state=s).fit(X).inertia_ for s in range(30)]
pd.DataFrame(res).plot.box(); plt.ylabel("inertia (меньше лучше)"); plt.title("Разброс результата из-за инициализации"); plt.show()
print(pd.DataFrame(res).describe().round(2))

## 3. Метод локтя

In [ ]:
ks = range(1, 11)
inertia = [KMeans(k, n_init=5, random_state=0).fit(X).inertia_ for k in ks]
plt.plot(ks, inertia, marker="o"); plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Метод локтя"); plt.show()

## 4. Картинка на 2D

In [ ]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(2, random_state=0); Z = svd.fit_transform(X); C = svd.transform(km.cluster_centers_)
plt.scatter(Z[:, 0], Z[:, 1], c=km.labels_, cmap="tab10", s=12, alpha=.7); plt.scatter(C[:, 0], C[:, 1], marker="X", s=200, c="black")
plt.title("K-Means: кластеры и центроиды (проекция SVD)"); plt.show()

## Задания

1. Найдите k, при котором ARI максимален. Совпадает ли оно с «локтем»?
2. Попробуйте `MiniBatchKMeans`. Насколько результат отличается и во сколько раз быстрее?
3. Почему для текстов полезно нормировать векторы (сферический K-Means)?